## Overview of the Python Project Structure

This Python folder contains the main code, notebooks, configuration files, and processing tools for the Classical Conditioning / Air-Wheel behavioral analysis project.

The project is organized into several main folders:

## 1. `config/`

This folder contains project-level configuration files.

The main file is:

```python
config/config.json
```

This file defines important paths:

```json
{
  "paths": {
    "code_base": "~/ccaw/Python",
    "raw_base": "/mnt/data/Classical_Conditioning",
    "proc_base": "/mnt/pdata/Classical_Conditioning"
  }
}
```

These paths tell Python where to find:

* the code base
* the raw data
* the processed data folder

This makes the code more flexible because paths can be changed in one place instead of editing many scripts.

## 2. `notebooks/`

This folder contains Jupyter notebooks used for interactive analysis.

Examples include:

```python
main_runner.ipynb
behavioral_characterization.ipynb
extract_LED_roi_signal.ipynb
```

These notebooks are useful for testing code, running analysis step-by-step, visualizing results, and developing new workflows before converting them into reusable scripts.

## 3. `proc/`

This folder contains processing modules.

Important files include:

```python
process_behavior_signals.py
behavior_metrics.py
organize.py
ami.py
```

These scripts are meant to process behavioral signals, calculate behavioral metrics, organize data, and run analysis routines.

For example:

```python
import proc.process_behavior_signals as pbs
```

allows the notebook to use functions from `process_behavior_signals.py`.

## 4. `src/utils/`

This folder contains utility functions that are used across the project.

Important files include:

```python
config.py
data_io.py
pdata_io.py
data_proc.py
organize_mp4s.py
```

These scripts handle common tasks such as:

* loading configuration paths
* reading raw data
* writing processed data
* organizing files
* creating directory structures

For example:

```python
import src.utils.config as config
import src.utils.data_io as dio
import src.utils.pdata_io as pdio
```

## 5. `scripts/`

This folder contains standalone scripts that can be run directly.

Examples include:

```python
process_h264.py
optical_flow.py
plot_OF_ME.py
export_session_data_to_matlab.py
```

These scripts are useful for specific tasks such as video processing, optical flow analysis, plotting motion energy, or exporting processed data to MATLAB.

## 6. `vis/`

This folder contains visualization and quality-control plotting functions.

Examples include:

```python
plots.py
plot_ami.py
qc_air_edges.py
qc_encoder_distance.py
```

These functions help visualize behavioral data and check whether signals such as air onset/offset and encoder distance were detected correctly.

## 7. `__pycache__/`

These folders are automatically created by Python.

They store compiled versions of Python files and do not need to be edited manually.

## Current Notebook Workflow

The notebook starts by enabling automatic reloading:

```python
%load_ext autoreload
%autoreload 2
```

This means that if you edit a Python file, the notebook will automatically reload the updated version.

Then the project path is added:

```python
repo_root = Path.cwd().parent
sys.path.append(str(repo_root))
```

This allows the notebook to import local project modules.

Next, important project modules are imported:

```python
import src.utils.config as config
import proc.process_behavior_signals as pbs
import src.utils.data_io as dio
import src.utils.pdata_io as pdio
```

Then the code loads the raw data location:

```python
data_root = config.RAW_BASE
```

and builds a dictionary of all animals, sessions, and related files:

```python
cc_data = dio.build_classical_conditioning_dict(data_root)
```

This `cc_data` dictionary is the main starting point for organizing the behavioral dataset. It should contain information about animals, sessions, videos, and associated data files.

Finally, the processed data location is defined:

```python
pdata_root = config.PROC_BASE
```

This path will be used later to save processed outputs.


In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

repo_root = Path.cwd().parent
sys.path.append(str(repo_root))

# import local libraries
import src.utils.config as config
import proc.process_behavior_signals as pbs
# import proc.organize as org
import src.utils.data_io as dio
import src.utils.pdata_io as pdio
import numpy as np

# print(config.PROJECT_ROOT)
# print(config.RAW_BASE)
# print(config.PROC_BASE)

import sys
from pathlib import Path

CODE_BASE = Path(config.CODE_BASE).expanduser()
sys.path.append(str(CODE_BASE))

print("Added to path:", CODE_BASE)

# make the data dictionary which contains list of all animals and sessions and
# file names of videos h264 and mat files

data_root = config.RAW_BASE # this variable now contains the path to the raw data directory, which is defined in the config file. It is used to build the classical conditioning data dictionary using the dio.build_classical_conditioning_dict function. The resulting cc_data variable will contain the structured information about the animals, sessions, and file names for the classical conditioning experiments.
cc_data = dio.build_classical_conditioning_dict(data_root)

pdata_root = config.PROC_BASE # this variable now contains the path to the processed data directory, which is defined in the config file. It is used to build the processed data dictionary using the pdio.build_processed_data_dict function. The resulting pdata variable will contain the structured information about the animals, sessions, and file names for the processed data of the classical conditioning experiments.
print(data_root)
print(pdata_root)

/home/nmldata2/ccaw/Python
Added to path: /home/nmldata2/ccaw/Python
/mnt/data/Classical_Conditioning
/mnt/pdata/Classical_Conditioning


In [3]:
print(cc_data)
print(cc_data["NML_04"]["2026_01_13"])

{'NML_04': {'2025_12_27': {'face': '/mnt/data/Classical_Conditioning/NML_04/2025_12_27/face_1440x1080_60_20251227_154519.h264', 'pupi': '/mnt/data/Classical_Conditioning/NML_04/2025_12_27/pupi_320x240_60_20251227_154519.h264', 'video': '/mnt/data/Classical_Conditioning/NML_04/2025_12_27/video_20251227_154511.h264', 'recording': '/mnt/data/Classical_Conditioning/NML_04/2025_12_27/recording_20251227_154502.mat', 'phase': 'habituation', 'path': '/mnt/data/Classical_Conditioning/NML_04/2025_12_27'}, '2025_12_28': {'face': '/mnt/data/Classical_Conditioning/NML_04/2025_12_28/face_1440x1080_60_20251228_154642.h264', 'pupi': '/mnt/data/Classical_Conditioning/NML_04/2025_12_28/pupi_320x240_60_20251228_154642.h264', 'video': '/mnt/data/Classical_Conditioning/NML_04/2025_12_28/video_20251228_154758.h264', 'recording': '/mnt/data/Classical_Conditioning/NML_04/2025_12_28/recording_20251228_154611.mat', 'phase': 'habituation', 'path': '/mnt/data/Classical_Conditioning/NML_04/2025_12_28'}, '2025_12_2

## Tutorial: Understanding `cc_data`

When we run:

```python
print(cc_data)
```

Python prints a large dictionary that contains information about all animals, all recording sessions, and all files found in the raw data folder.

Think of `cc_data` as a table of contents for the experiment.

## 1. Main Structure

The structure is:

```python
cc_data[animal][session][file_type]
```

For example:

```python
cc_data["NML_04"]["2025_12_27"]["face"]
```

This gives the path to the face video for animal `NML_04` on session date `2025_12_27`.

## 2. First Level: Animal

The first level contains animal IDs:

```python
"NML_04"
"NML_05"
```

Each animal has its own set of recording sessions.

## 3. Second Level: Session Date

Inside each animal, the data are organized by date:

```python
"2025_12_27"
"2025_12_28"
"2026_01_01"
```

Each date represents one recording session.

## 4. Third Level: Session Information

Each session contains file paths and metadata:

```python
{
    "face": "...face video...",
    "pupi": "...pupil video...",
    "video": "...main/body video...",
    "recording": "...MAT recording file...",
    "phase": "habituation",
    "path": "...session folder..."
}
```

## 5. Meaning of Each Field

### `face`

This is the face-camera video.

It may be used to analyze:

```python
facial motion
whisker movement
nose movement
grooming
face-related optical flow
```

### `pupi`

This is the pupil-camera video.

It may be used to analyze:

```python
pupil size
eye movement
blinking
arousal-related changes
```

### `video`

This is the main behavioral video.

It may be used to analyze:

```python
body movement
paw movement
wheel movement
locomotion
motion energy
```

### `recording`

This is the `.mat` file from the recording system.

It may contain:

```python
air signal
encoder signal
TTL signals
timing information
synchronization signals
```

### `phase`

This tells us which experimental phase the session belongs to.

Examples:

```python
"habituation"
"air_training"
"unknown"
```

This is useful because we can analyze only one phase at a time.

### `path`

This is the folder path for that session.

It tells Python where that session is stored.

## 6. Why This Dictionary Is Useful

Instead of manually writing file paths one by one, we can let Python find and organize the files automatically.

For example, we can loop through all animals and sessions:

```python
for animal in cc_data:
    for session in cc_data[animal]:
        print(animal, session)
```

We can also select only habituation sessions:

```python
for animal in cc_data:
    for session in cc_data[animal]:
        if cc_data[animal][session]["phase"] == "habituation":
            print(animal, session)
```

Or get one specific file:

```python
face_file = cc_data["NML_04"]["2025_12_27"]["face"]
```

## 7. Simple Explanation

In simple words:

`cc_data` is a structured map of the experiment.

It tells us:

```python
which animals were recorded
which days they were recorded
which files belong to each day
which experimental phase each day belongs to
where the files are located
```

This makes the analysis easier, cleaner, and less error-prone.
